# 05：第二轮数据改进与三组对照

04 已完成第一轮训练。下一步保留目标写法，重点改善 **动作遗漏、直接台词归属和重复句子**。

agent 已准备 50 条训练 / 12 条验证，包含第一轮数据和 26 条新增样本；另有 10 条训练前固定的未见测试，参考答案不进入训练或推理输入。本轮数据独立导出到 `datasets/processed/teaching-v2/`。

按 **步骤 1 → 2 → 3** 执行，选择 `Python (Text LoRA - Qwen)`。你的主要工作是监督新增答案、决定采用本轮方案、审阅输出。步骤 2 的一个 Cell 自动完成训练和评估；数据编写、参数配置和报告整理已由 agent 处理。

当前交付状态：数据与 CPU 模板检查已通过，第二轮 GPU 训练尚未执行；是否改善内容，要等第二轮实际生成后评审。

In [1]:
import html
import json
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

from IPython.display import HTML, Markdown, display

project_root = next((candidate for parent in [Path.cwd(), *Path.cwd().parents]
                     for candidate in (parent, parent / "text-lora")
                     if (candidate / "scripts/check_environment.py").is_file()), None)
if project_root is None:
    raise RuntimeError("请从当前 text-lora 项目打开 Notebook")
if str(project_root / "scripts") not in sys.path:
    sys.path.insert(0, str(project_root / "scripts"))
from check_environment import check_python
from prepare_training_data import canonical_hash, dataset_paths, validate_corpus
from run_second_iteration import CORPUS_V2, LATEST, current_state

check_python()


def run_script(script, *arguments):
    stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S_%fZ")
    log = project_root / "outputs/logs" / (Path(script).stem + "-" + stamp + ".log")
    log.parent.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, "-u", str(project_root / "scripts" / script), *map(str, arguments)]
    print("执行：", script, *arguments)
    print("完整日志：", log)
    process = subprocess.Popen(command, cwd=project_root, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        with log.open("x", encoding="utf-8") as file:
            for line in process.stdout:
                file.write(line)
                file.flush()
                if any(tag in line for tag in ("[RUN]", "[PASS]", "{'loss':", "{'eval_loss':")):
                    print(line.strip(), flush=True)
        code = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    if code:
        print(log.read_text(encoding="utf-8")[-12000:])
        raise RuntimeError(f"{script} 失败；完整日志：{log}，交给 agent 修复")
    return log


Python：3.12.13
解释器：/home/muyi086/github/test_LLM/text-lora/.venv/bin/python3


## 步骤 1：监督新增数据与查看预检查

只展开新增的 26 条，不要求重新审阅第一轮全部样本。动作检查项要求放在旁白；直接话语检查项要求正确说话者和正确次数。它们是人工声明的事实清单，程序不能自动证明所有语义都正确。

数据与两个固定评估集均做来源分离、重复和文本相似度检查。CPU 检查不加载模型权重。沿用第一轮的 rank、学习率、3 轮、512 长度等设置；新增数据意味着优化步数也会增加。

In [2]:
run_script("prepare_training_data.py", "--corpus", CORPUS_V2, "--check-tokens")
samples, _ = validate_corpus(CORPUS_V2)
checks = json.loads(dataset_paths(CORPUS_V2)["checks"].read_text())
print("训练/验证：", sum(s["split"] == "train" for s in samples), sum(s["split"] == "val" for s in samples))
print("完整模板最大 tokens：", max(x["total_tokens"] for x in checks["lengths"]))
blocks = []
for sample in samples:
    if not sample["id"].startswith("r2_"):
        continue
    title = f"{sample['split']} / {sample['id']} / {sample['category']}"
    blocks.append("<details><summary>" + html.escape(title) + "</summary><p>原文：</p><pre>"
                  + html.escape(sample["source_text"]) + "</pre><p>答案：</p><pre>"
                  + html.escape(sample["messages"][-1]["content"]) + "</pre><p>核对："
                  + html.escape("；".join(sample["review_criteria"])) + "</p></details>")
display(HTML("".join(blocks)))
run_script("train_lora.py", "--corpus", CORPUS_V2, "--mode", "full", "--dry-run")
run_script("run_second_iteration.py", "--dry-run")


执行： prepare_training_data.py --corpus /home/muyi086/github/test_LLM/text-lora/datasets/teaching_samples_v2.json --check-tokens
完整日志： /home/muyi086/github/test_LLM/text-lora/outputs/logs/prepare_training_data-20261008T135424_082060Z.log


[PASS] 50 条训练 + 12 条验证；pilot 保留；来源、格式和测试隔离通过。


训练/验证： 50 12
完整模板最大 tokens： 235


执行： train_lora.py --corpus /home/muyi086/github/test_LLM/text-lora/datasets/teaching_samples_v2.json --mode full --dry-run
完整日志： /home/muyi086/github/test_LLM/text-lora/outputs/logs/train_lora-20261008T135430_798565Z.log


[PASS] dry run；未实例化 SftArguments、未加载权重、未创建运行目录。


执行： run_second_iteration.py --dry-run
完整日志： /home/muyi086/github/test_LLM/text-lora/outputs/logs/run_second_iteration-20261008T135434_943228Z.log


[PASS] 计划：CPU模板检查 → 3步试跑 → 3轮全新LoRA → 原10条第二轮推理 → 新10条原模型/第一轮/第二轮推理 → 两份三组对照。未启动GPU任务。


PosixPath('/home/muyi086/github/test_LLM/text-lora/outputs/logs/run_second_iteration-20261008T135434_943228Z.log')

## 步骤 2：采用本轮方案后，运行一次完整实验

下面这个 Cell 会真实使用 GPU，依次自动完成：

1. CPU 模板检查、3 步试跑、从原模型重新初始化的 3 轮 LoRA 训练。
2. 原有 10 条测试的第二轮输出；复用原模型和第一轮的原有输出。
3. 新 10 条未见测试分别运行原模型、第一轮 LoRA、第二轮 LoRA。
4. 核对输入、模型、依赖和解码设置，生成原有测试与新测试两份三组对照。

按独立验证 loss 选择 checkpoint；参考答案和固定测试不用于选 checkpoint。预计需数分钟，以实际日志为准。先停止你自己的其他 GPU 模型服务。失败时保留具体阶段、日志和已完成结果，由 agent 处理。

再次运行会复用完整且数据/配置匹配的第二轮结果。第一轮数据、Notebook 03/04 与 checkpoint 保留。

In [ ]:
# 执行此 Cell 即启动第二轮完整实验，无需手动逐条运行脚本。
run_script("run_second_iteration.py")


## 步骤 3：查看内容是否改善，并作下一轮决策

优先查看 **新 10 条未见测试**，再看原有测试。双标签比例仅衡量格式；动作完整、直接台词归属、重复次数与普通任务表现都要查看实际输出。参考稿只是保守写法之一，不能把与参考逐字不同直接算成语义错误。

两轮验证集合不同，验证 loss 的绝对值不能直接当作改进比例。两轮都训练 3 轮，但第二轮数据更多、优化步数更多；这是整体迭代对照，不是只隔离某一个变量的因果实验。

In [3]:
if not LATEST.exists():
    print("当前状态：第二轮训练待执行；数据和预检查已完成。采用方案后执行步骤 2，或交给 agent 执行。")
else:
    receipt = json.loads(LATEST.read_text())
    root = Path(receipt["run_dir"])
    metadata = json.loads((root / "iteration_metadata.json").read_text())
    if metadata["status"] != "complete" or metadata["state"] != current_state(CORPUS_V2):
        raise RuntimeError("已有第二轮结果与当前数据/配置不匹配，请交给 agent 核对")
    training = json.loads((root / "train-full/run_metadata.json").read_text())
    print("第二轮目录：", root)
    print("优化步数：", training["result"]["global_step"])
    print("PyTorch 峰值 allocated/reserved GiB：", training["peak_allocated_gib"], training["peak_reserved_gib"])
    print("选中的 adapter：", metadata["second_adapter"])
    display(Markdown("### 新 10 条未见测试"))
    display(Markdown((Path(metadata["fresh_comparison_dir"]) / "comparison.md").read_text()))
    display(Markdown("### 原有 10 条测试"))
    display(Markdown((Path(metadata["fixed_comparison_dir"]) / "comparison.md").read_text()))


当前状态：第二轮训练待执行；数据和预检查已完成。采用方案后执行步骤 2，或交给 agent 执行。


你主要决定是否沿用写法、是否采用第二轮数据、下一轮重点解决哪些实际错误。样本修订、训练执行和逐条内容评审由 agent 完成。训练后先看内容改善与普通任务表现，再决定是否继续扩充；第一轮 checkpoint 留作对照。

细节见 `datasets/iteration2_review.md`。